# SimFin Data Retrieval


## 1. Setup

In [23]:
import simfin as sf
import pandas as pd
import numpy as np

sf.set_api_key('eaf6d370-a27d-44df-86f2-713f5a2f6a5b')
sf.set_data_dir('~/simfin_data/')
print("SimFin version:", sf.__version__)


SimFin version: 1.0.2


## 2. Companies, Industries, Markets

In [2]:
df_companies = sf.load_companies(market='us')
df_companies.reset_index(inplace=True)
print(f"Companies: {len(df_companies):,}")
df_companies.head(3)


Dataset "us-companies" on disk (0 days old).
- Loading from disk ... Done!
Companies: 6,585


,Ticker,SimFinId,Company Name,IndustryId,ISIN,End of financial year (month),Number Employees,Business Summary,Market,CIK,Main Currency
0,A,45846,AGILENT TECHNOLOGIES INC,106001.0,US00846U1016,10.0,16400.0,Agilent Technologies Inc is engaged in life sc...,us,1090872.0,USD
1,A21,1333027,Li Auto Inc.,NaN,NaN,12.0,NaN,NaN,us,1791706.0,USD
2,AA,367153,Alcoa Corp,110004.0,US0138721065,12.0,12900.0,Alcoa Corp is an integrated aluminum company. ...,us,1675149.0,USD


In [3]:
df_industries = sf.load_industries()
if df_industries.index.name == 'IndustryId':
    df_industries = df_industries.reset_index()
df_industries['IndustryId'] = pd.to_numeric(df_industries['IndustryId'], errors='coerce')
print(f"Industries: {len(df_industries):,}")
df_industries.head(3)


Dataset "industries" on disk (0 days old).
- Loading from disk ... Done!
Industries: 74


,IndustryId,Industry,Sector
0,100001,Industrial Products,Industrials
1,100002,Business Services,Industrials
2,100003,Engineering & Construction,Industrials


In [4]:
df_markets = sf.load_markets()
print(df_markets)


Dataset "markets" on disk (0 days old).
- Loading from disk ... Done!
            Market Name Currency
MarketId                        
ca               Canada      CAD
cn                China      CNY
de              Germany      EUR
us        United States      USD


In [5]:
# Attach sector/industry to companies
df_companies['IndustryId'] = pd.to_numeric(df_companies['IndustryId'], errors='coerce')
df_companies = df_companies.merge(
    df_industries[['IndustryId', 'Sector', 'Industry']],
    on='IndustryId', how='left'
)
print(f"Companies with Sector: {df_companies['Sector'].notna().sum():,} / {len(df_companies):,}")
df_companies[['Company Name', 'Sector', 'Industry']].dropna().head(5)


Companies with Sector: 6,273 / 6,585


,Company Name,Sector,Industry
0,AGILENT TECHNOLOGIES INC,Healthcare,Medical Diagnostics & Research
2,Alcoa Corp,Basic Materials,Metals & Mining
3,Ares Acquisition Corporation,Financial Services,Banks
4,Artius II Acquisition Inc. Class A Ordinary Sh...,Financial Services,Banks
5,ATA Creativity Global,Consumer Defensive,Education


## 3. Income Statements

In [6]:
df_income      = sf.load_income(variant='annual', market='us')
df_income_banks = sf.load_income_banks(variant='annual', market='us')
df_income_ins   = sf.load_income_insurance(variant='annual', market='us')

df_income['fin_type']       = 'standard'
df_income_banks['fin_type'] = 'bank'
df_income_ins['fin_type']   = 'insurance'

df_income_all = pd.concat([df_income, df_income_banks, df_income_ins], axis=0, sort=False)
print(f"Income — standard: {df_income.shape}, banks: {df_income_banks.shape}, insurance: {df_income_ins.shape}")
print(f"Combined:          {df_income_all.shape}")
df_income_all.head(2)


Dataset "us-income-annual" on disk (0 days old).
- Loading from disk ... Done!
Dataset "us-income-banks-annual" on disk (0 days old).
- Loading from disk ... Done!
Dataset "us-income-insurance-annual" on disk (0 days old).
- Loading from disk ... Done!
Income — standard: (16907, 27), banks: (863, 21), insurance: (300, 19)
Combined:          (18070, 32)


SimFinId Currency  Fiscal Year Fiscal Period Publish Date  \
Ticker Report Date                                                              
A      2020-10-31      45846      USD         2020            FY   2020-12-18   
       2021-10-31      45846      USD         2021            FY   2021-12-17   

                   Restated Date  Shares (Basic)  Shares (Diluted)  \
Ticker Report Date                                                   
A      2020-10-31     2022-12-21     309000000.0       312000000.0   
       2021-10-31     2023-12-20     305000000.0       307000000.0   

                         Revenue  Cost of Revenue  ...  \
Ticker Report Date                                 ...   
A      2020-10-31   5.339000e+09    -2.502000e+09  ...   
       2021-10-31   6.319000e+09    -2.912000e+09  ...   

                    Income (Loss) from Continuing Operations  \
Ticker Report Date                                             
A      2020-10-31                                  719000000   
       2021-10-31                                 1210000000   

                    Net Extraordinary Gains (Losses)  Net Income  \
Ticker Report Date                                                 
A      2020-10-31                                NaN   719000000   
       2021-10-31                                NaN  1210000000   

                    Net Income (Common)  fin_type  Provision for Loan Losses  \
Ticker Report Date                                                             
A      2020-10-31             719000000  standard                        NaN   
       2021-10-31            1210000000  standard                        NaN   

                    Net Revenue after Provisions  Total Non-Interest Expense  \
Ticker Report Date                                                             
A      2020-10-31                            NaN                         NaN   
       2021-10-31                            NaN                         NaN   

                    Total Claims & Losses  \
Ticker Report Date                          
A      2020-10-31                     NaN   
       2021-10-31                     NaN   

                    Income (Loss) from Affiliates, Net of Taxes  
Ticker Report Date                                               
A      2020-10-31                                           NaN  
       2021-10-31                                           NaN  

[2 rows x 32 columns]

In [7]:
# ── Missing values: Combined Income Statements ──────────────────────────────────────────────────
_num_cols = df_income_all.select_dtypes(include='number').columns.tolist()
_miss = df_income_all[_num_cols].isnull().mean().sort_values(ascending=False)
_miss = _miss[_miss > 0]

print(f"Combined Income Statements")
print(f"  Shape:                {str(df_income_all.shape)}")
print(f"  Numeric columns:      {len(_num_cols)}")
print(f"  Columns with missing: {len(_miss)}")
print(f"  Overall missing rate: {(df_income_all[_num_cols].isnull().mean().mean()*100):.1f}%")
if len(_miss):
    print(f"  Top 10 most incomplete columns:")
    for col, pct in _miss.head(10).items():
        print(f"    {col:<55} {pct*100:5.1f}%")


Combined Income Statements
  Shape:                (18070, 32)
  Numeric columns:      27
  Columns with missing: 22
  Overall missing rate: 32.2%
  Top 10 most incomplete columns:
    Income (Loss) from Affiliates, Net of Taxes              99.9%
    Total Claims & Losses                                    98.6%
    Provision for Loan Losses                                95.5%
    Total Non-Interest Expense                               95.3%
    Net Revenue after Provisions                             95.2%
    Net Extraordinary Gains (Losses)                         91.9%
    Depreciation & Amortization                              62.9%
    Research & Development                                   55.4%
    Abnormal Gains (Losses)                                  39.5%
    Gross Profit                                             27.1%


## 4. Balance Sheets

In [8]:
df_balance      = sf.load_balance(variant='annual', market='us')
df_balance_banks = sf.load_balance_banks(variant='annual', market='us')
df_balance_ins   = sf.load_balance_insurance(variant='annual', market='us')

df_balance['fin_type']       = 'standard'
df_balance_banks['fin_type'] = 'bank'
df_balance_ins['fin_type']   = 'insurance'

df_balance_all = pd.concat([df_balance, df_balance_banks, df_balance_ins], axis=0, sort=False)
print(f"Balance — standard: {df_balance.shape}, banks: {df_balance_banks.shape}, insurance: {df_balance_ins.shape}")
print(f"Combined:           {df_balance_all.shape}")
df_balance_all.head(2)


Dataset "us-balance-annual" on disk (0 days old).
- Loading from disk ... 

Done!
Dataset "us-balance-banks-annual" on disk (0 days old).
- Loading from disk ... Done!
Dataset "us-balance-insurance-annual" on disk (0 days old).
- Loading from disk ... Done!
Balance — standard: (16908, 29), banks: (863, 26), insurance: (299, 25)
Combined:           (18070, 38)


SimFinId Currency  Fiscal Year Fiscal Period Publish Date  \
Ticker Report Date                                                              
A      2020-10-31      45846      USD         2020            Q4   2020-12-18   
       2021-10-31      45846      USD         2021            Q4   2021-12-17   

                   Restated Date  Shares (Basic)  Shares (Diluted)  \
Ticker Report Date                                                   
A      2020-10-31     2021-12-17     309000000.0       312000000.0   
       2021-10-31     2022-12-21     305000000.0       307000000.0   

                    Cash, Cash Equivalents & Short Term Investments  \
Ticker Report Date                                                    
A      2020-10-31                                      1.441000e+09   
       2021-10-31                                      1.575000e+09   

                    Accounts & Notes Receivable  ...  fin_type  \
Ticker Report Date                               ...             
A      2020-10-31                  1.038000e+09  ...  standard   
       2021-10-31                  1.172000e+09  ...  standard   

                    Interbank Assets  Short & Long Term Investments  \
Ticker Report Date                                                    
A      2020-10-31                NaN                            NaN   
       2021-10-31                NaN                            NaN   

                    Net Loans  Net Fixed Assets  Total Deposits  \
Ticker Report Date                                                
A      2020-10-31         NaN               NaN             NaN   
       2021-10-31         NaN               NaN             NaN   

                    Preferred Equity  Total Investments  Insurance Reserves  \
Ticker Report Date                                                            
A      2020-10-31                NaN                NaN                 NaN   
       2021-10-31                NaN                NaN                 NaN   

                    Policyholders Equity  
Ticker Report Date                        
A      2020-10-31                    NaN  
       2021-10-31                    NaN  

[2 rows x 38 columns]

In [9]:
# ── Missing values: Combined Balance Sheets ──────────────────────────────────────────────────
_num_cols = df_balance_all.select_dtypes(include='number').columns.tolist()
_miss = df_balance_all[_num_cols].isnull().mean().sort_values(ascending=False)
_miss = _miss[_miss > 0]

print(f"Combined Balance Sheets")
print(f"  Shape:                {str(df_balance_all.shape)}")
print(f"  Numeric columns:      {len(_num_cols)}")
print(f"  Columns with missing: {len(_miss)}")
print(f"  Overall missing rate: {(df_balance_all[_num_cols].isnull().mean().mean()*100):.1f}%")
if len(_miss):
    print(f"  Top 10 most incomplete columns:")
    for col, pct in _miss.head(10).items():
        print(f"    {col:<55} {pct*100:5.1f}%")


Combined Balance Sheets
  Shape:                (18070, 38)
  Numeric columns:      33
  Columns with missing: 28
  Overall missing rate: 37.4%
  Top 10 most incomplete columns:
    Policyholders Equity                                    100.0%
    Preferred Equity                                         98.6%
    Insurance Reserves                                       98.5%
    Total Investments                                        98.4%
    Interbank Assets                                         97.4%
    Net Loans                                                95.7%
    Net Fixed Assets                                         95.4%
    Total Deposits                                           95.4%
    Short & Long Term Investments                            95.3%
    Long Term Investments & Receivables                      78.3%


## 5. Cash Flow Statements

In [10]:
df_cashflow      = sf.load_cashflow(variant='annual', market='us')
df_cashflow_banks = sf.load_cashflow_banks(variant='annual', market='us')
df_cashflow_ins   = sf.load_cashflow_insurance(variant='annual', market='us')

df_cashflow['fin_type']       = 'standard'
df_cashflow_banks['fin_type'] = 'bank'
df_cashflow_ins['fin_type']   = 'insurance'

df_cashflow_all = pd.concat([df_cashflow, df_cashflow_banks, df_cashflow_ins], axis=0, sort=False)
print(f"Cashflow — standard: {df_cashflow.shape}, banks: {df_cashflow_banks.shape}, insurance: {df_cashflow_ins.shape}")
print(f"Combined:            {df_cashflow_all.shape}")
df_cashflow_all.head(2)


Dataset "us-cashflow-annual" on disk (0 days old).
- Loading from disk ... Done!
Dataset "us-cashflow-banks-annual" on disk (0 days old).
- Loading from disk ... Done!
Dataset "us-cashflow-insurance-annual" on disk (0 days old).
- Loading from disk ... Done!
Cashflow — standard: (16905, 27), banks: (863, 25), insurance: (300, 22)
Combined:            (18068, 31)


SimFinId Currency  Fiscal Year Fiscal Period Publish Date  \
Ticker Report Date                                                              
A      2020-10-31      45846      USD         2020            FY   2020-12-18   
       2021-10-31      45846      USD         2021            FY   2021-12-17   

                   Restated Date  Shares (Basic)  Shares (Diluted)  \
Ticker Report Date                                                   
A      2020-10-31     2022-12-21     309000000.0       312000000.0   
       2021-10-31     2023-12-20     305000000.0       307000000.0   

                    Net Income/Starting Line  Depreciation & Amortization  \
Ticker Report Date                                                          
A      2020-10-31               7.190000e+08                  308000000.0   
       2021-10-31               1.210000e+09                  321000000.0   

                    ...  Dividends Paid  Cash from (Repayment of) Debt  \
Ticker Report Date  ...                                                  
A      2020-10-31   ...    -222000000.0                    -45000000.0   
       2021-10-31   ...    -236000000.0                    356000000.0   

                    Cash from (Repurchase of) Equity  \
Ticker Report Date                                     
A      2020-10-31                       -409000000.0   
       2021-10-31                       -733000000.0   

                    Net Cash from Financing Activities  Net Change in Cash  \
Ticker Report Date                                                           
A      2020-10-31                         -717000000.0            59000000   
       2021-10-31                         -696000000.0            43000000   

                    fin_type  Provision for Loan Losses  \
Ticker Report Date                                        
A      2020-10-31   standard                        NaN   
       2021-10-31   standard                        NaN   

                    Net Change in Loans & Interbank  \
Ticker Report Date                                    
A      2020-10-31                               NaN   
       2021-10-31                               NaN   

                    Effect of Foreign Exchange Rates  \
Ticker Report Date                                     
A      2020-10-31                                NaN   
       2021-10-31                                NaN   

                    Net Change in Investments  
Ticker Report Date                             
A      2020-10-31                         NaN  
       2021-10-31                         NaN  

[2 rows x 31 columns]

In [11]:
# ── Missing values: Combined Cash Flow Statements ──────────────────────────────────────────────────
_num_cols = df_cashflow_all.select_dtypes(include='number').columns.tolist()
_miss = df_cashflow_all[_num_cols].isnull().mean().sort_values(ascending=False)
_miss = _miss[_miss > 0]

print(f"Combined Cash Flow Statements")
print(f"  Shape:                {str(df_cashflow_all.shape)}")
print(f"  Numeric columns:      {len(_num_cols)}")
print(f"  Columns with missing: {len(_miss)}")
print(f"  Overall missing rate: {(df_cashflow_all[_num_cols].isnull().mean().mean()*100):.1f}%")
if len(_miss):
    print(f"  Top 10 most incomplete columns:")
    for col, pct in _miss.head(10).items():
        print(f"    {col:<55} {pct*100:5.1f}%")


Combined Cash Flow Statements
  Shape:                (18068, 31)
  Numeric columns:      26
  Columns with missing: 23
  Overall missing rate: 40.1%
  Top 10 most incomplete columns:
    Change in Inventories                                    99.5%
    Change in Accounts Payable                               99.4%
    Change in Accounts Receivable                            99.3%
    Change in Other                                          99.1%
    Effect of Foreign Exchange Rates                         99.0%
    Net Change in Investments                                98.4%
    Provision for Loan Losses                                96.0%
    Net Change in Loans & Interbank                          95.5%
    Net Change in Long Term Investment                       66.8%
    Net Cash from Acquisitions & Divestitures                62.1%


## 6. Share Prices

In [12]:
df_prices = sf.load_shareprices(variant='daily', market='us')
print(f"Share prices (daily): {df_prices.shape}")
df_prices.head(3)


Dataset "us-shareprices-daily" on disk (0 days old).
- Loading from disk ... Done!
Share prices (daily): (6250153, 9)


SimFinId   Open   High    Low  Close  Adj. Close   Volume  \
Ticker Date                                                                    
A      2020-07-06     45846  89.02  90.64  89.02  89.31       85.85  1412243   
       2020-07-07     45846  88.84  90.04  88.67  89.21       85.75  1441704   
       2020-07-08     45846  89.52  90.33  89.13  89.54       86.07  1069594   

                   Dividend  Shares Outstanding  
Ticker Date                                      
A      2020-07-06       NaN         308777396.0  
       2020-07-07       NaN         308777396.0  
       2020-07-08       NaN         308777396.0

In [13]:
# ── Missing values: Daily Share Prices ──────────────────────────────────────────────────
_num_cols = df_prices.select_dtypes(include='number').columns.tolist()
_miss = df_prices[_num_cols].isnull().mean().sort_values(ascending=False)
_miss = _miss[_miss > 0]

print(f"Daily Share Prices")
print(f"  Shape:                {str(df_prices.shape)}")
print(f"  Numeric columns:      {len(_num_cols)}")
print(f"  Columns with missing: {len(_miss)}")
print(f"  Overall missing rate: {(df_prices[_num_cols].isnull().mean().mean()*100):.1f}%")
if len(_miss):
    print(f"  Top 10 most incomplete columns:")
    for col, pct in _miss.head(10).items():
        print(f"    {col:<55} {pct*100:5.1f}%")


Daily Share Prices
  Shape:                (6250153, 9)
  Numeric columns:      9
  Columns with missing: 2
  Overall missing rate: 11.9%
  Top 10 most incomplete columns:
    Dividend                                                 99.4%
    Shares Outstanding                                        8.0%


## 7. Merge All Fundamentals

In [14]:
DROP_COLS = ['SimFinId', 'Currency', 'Fiscal Period', 'Publish Date',
             'Restated Date', 'Shares (Basic)', 'Shares (Diluted)']

def clean(df):
    return df.drop(columns=[c for c in DROP_COLS if c in df.columns])

df_fund = (
    clean(df_income_all)
    .join(clean(df_balance_all),  how='outer', lsuffix='', rsuffix='_bal')
    .join(clean(df_cashflow_all), how='outer', lsuffix='', rsuffix='_cf')
)

df_fund.index.names = ['Ticker', 'Report Date']
print(f"Merged fundamentals: {df_fund.shape}")
df_fund.head(2)


Merged fundamentals: (18088, 80)


Fiscal Year       Revenue  Cost of Revenue  Gross Profit  \
Ticker Report Date                                                             
A      2020-10-31        2020.0  5.339000e+09    -2.502000e+09  2.837000e+09   
       2021-10-31        2021.0  6.319000e+09    -2.912000e+09  3.407000e+09   

                    Operating Expenses  Selling, General & Administrative  \
Ticker Report Date                                                          
A      2020-10-31        -1.991000e+09                      -1.496000e+09   
       2021-10-31        -2.060000e+09                      -1.619000e+09   

                    Research & Development  Depreciation & Amortization  \
Ticker Report Date                                                        
A      2020-10-31             -495000000.0                          NaN   
       2021-10-31             -441000000.0                          NaN   

                    Operating Income (Loss)  Non-Operating Income (Loss)  ...  \
Ticker Report Date                                                        ...   
A      2020-10-31              8.460000e+08                   -4000000.0  ...   
       2021-10-31              1.347000e+09                   13000000.0  ...   

                    Dividends Paid  Cash from (Repayment of) Debt  \
Ticker Report Date                                                  
A      2020-10-31     -222000000.0                    -45000000.0   
       2021-10-31     -236000000.0                    356000000.0   

                    Cash from (Repurchase of) Equity  \
Ticker Report Date                                     
A      2020-10-31                       -409000000.0   
       2021-10-31                       -733000000.0   

                    Net Cash from Financing Activities  Net Change in Cash  \
Ticker Report Date                                                           
A      2020-10-31                         -717000000.0          59000000.0   
       2021-10-31                         -696000000.0          43000000.0   

                    fin_type_cf  Provision for Loan Losses_cf  \
Ticker Report Date                                              
A      2020-10-31      standard                           NaN   
       2021-10-31      standard                           NaN   

                    Net Change in Loans & Interbank  \
Ticker Report Date                                    
A      2020-10-31                               NaN   
       2021-10-31                               NaN   

                    Effect of Foreign Exchange Rates Net Change in Investments  
Ticker Report Date                                                              
A      2020-10-31                                NaN                       NaN  
       2021-10-31                                NaN                       NaN  

[2 rows x 80 columns]

In [15]:
# ── Missing values: Merged Fundamentals Panel ──────────────────────────────────────────────────
_num_cols = df_fund.select_dtypes(include='number').columns.tolist()
_miss = df_fund[_num_cols].isnull().mean().sort_values(ascending=False)
_miss = _miss[_miss > 0]

print(f"Merged Fundamentals Panel")
print(f"  Shape:                {str(df_fund.shape)}")
print(f"  Numeric columns:      {len(_num_cols)}")
print(f"  Columns with missing: {len(_miss)}")
print(f"  Overall missing rate: {(df_fund[_num_cols].isnull().mean().mean()*100):.1f}%")
if len(_miss):
    print(f"  Top 10 most incomplete columns:")
    for col, pct in _miss.head(10).items():
        print(f"    {col:<55} {pct*100:5.1f}%")


Merged Fundamentals Panel
  Shape:                (18088, 80)
  Numeric columns:      77
  Columns with missing: 77
  Overall missing rate: 40.8%
  Top 10 most incomplete columns:
    Policyholders Equity                                    100.0%
    Income (Loss) from Affiliates, Net of Taxes              99.9%
    Change in Inventories                                    99.5%
    Change in Accounts Payable                               99.4%
    Change in Accounts Receivable                            99.3%
    Change in Other                                          99.1%
    Effect of Foreign Exchange Rates                         99.0%
    Preferred Equity                                         98.6%
    Total Claims & Losses                                    98.6%
    Insurance Reserves                                       98.5%


In [16]:
# Attach company metadata
company_meta = df_companies[['Ticker', 'Company Name', 'IndustryId', 'Sector', 'Industry',
                              'Market', 'Main Currency']].copy()
company_meta.index = company_meta.index.astype(str)

df_fund = df_fund.reset_index()
df_fund['Ticker'] = df_fund['Ticker'].astype(str)
df_fund = df_fund.merge(
    company_meta,
    on='Ticker', how='left'
)
df_fund = df_fund.set_index(['Ticker', 'Report Date'])

# USD-denominated companies only
df_fund = df_fund[df_fund['Main Currency'].fillna('USD') == 'USD']
print(f"After USD filter: {df_fund.shape}")
print(f"Unique companies: {df_fund.index.get_level_values('Ticker').nunique():,}")
df_fund.head(3)


After USD filter: (18088, 86)
Unique companies: 4,787


Fiscal Year       Revenue  Cost of Revenue  Gross Profit  \
Ticker Report Date                                                             
A      2020-10-31        2020.0  5.339000e+09    -2.502000e+09  2.837000e+09   
       2021-10-31        2021.0  6.319000e+09    -2.912000e+09  3.407000e+09   
       2022-10-31        2022.0  6.848000e+09    -3.126000e+09  3.722000e+09   

                    Operating Expenses  Selling, General & Administrative  \
Ticker Report Date                                                          
A      2020-10-31        -1.991000e+09                      -1.496000e+09   
       2021-10-31        -2.060000e+09                      -1.619000e+09   
       2022-10-31        -2.104000e+09                      -1.637000e+09   

                    Research & Development  Depreciation & Amortization  \
Ticker Report Date                                                        
A      2020-10-31             -495000000.0                          NaN   
       2021-10-31             -441000000.0                          NaN   
       2022-10-31             -467000000.0                          NaN   

                    Operating Income (Loss)  Non-Operating Income (Loss)  ...  \
Ticker Report Date                                                        ...   
A      2020-10-31              8.460000e+08                   -4000000.0  ...   
       2021-10-31              1.347000e+09                   13000000.0  ...   
       2022-10-31              1.618000e+09                 -114000000.0  ...   

                    Provision for Loan Losses_cf  \
Ticker Report Date                                 
A      2020-10-31                            NaN   
       2021-10-31                            NaN   
       2022-10-31                            NaN   

                    Net Change in Loans & Interbank  \
Ticker Report Date                                    
A      2020-10-31                               NaN   
       2021-10-31                               NaN   
       2022-10-31                               NaN   

                    Effect of Foreign Exchange Rates  \
Ticker Report Date                                     
A      2020-10-31                                NaN   
       2021-10-31                                NaN   
       2022-10-31                                NaN   

                    Net Change in Investments              Company Name  \
Ticker Report Date                                                        
A      2020-10-31                         NaN  AGILENT TECHNOLOGIES INC   
       2021-10-31                         NaN  AGILENT TECHNOLOGIES INC   
       2022-10-31                         NaN  AGILENT TECHNOLOGIES INC   

                    IndustryId      Sector                        Industry  \
Ticker Report Date                                                           
A      2020-10-31     106001.0  Healthcare  Medical Diagnostics & Research   
       2021-10-31     106001.0  Healthcare  Medical Diagnostics & Research   
       2022-10-31     106001.0  Healthcare  Medical Diagnostics & Research   

                    Market Main Currency  
Ticker Report Date                        
A      2020-10-31       us           USD  
       2021-10-31       us           USD  
       2022-10-31       us           USD  

[3 rows x 86 columns]

In [17]:
# ── Missing values: Merged Panel + Company Metadata ──────────────────────────────────────────────────
_num_cols = df_fund.select_dtypes(include='number').columns.tolist()
_miss = df_fund[_num_cols].isnull().mean().sort_values(ascending=False)
_miss = _miss[_miss > 0]

print(f"Merged Panel + Company Metadata")
print(f"  Shape:                {str(df_fund.shape)}")
print(f"  Numeric columns:      {len(_num_cols)}")
print(f"  Columns with missing: {len(_miss)}")
print(f"  Overall missing rate: {(df_fund[_num_cols].isnull().mean().mean()*100):.1f}%")
if len(_miss):
    print(f"  Top 10 most incomplete columns:")
    for col, pct in _miss.head(10).items():
        print(f"    {col:<55} {pct*100:5.1f}%")


Merged Panel + Company Metadata
  Shape:                (18088, 86)
  Numeric columns:      78
  Columns with missing: 78
  Overall missing rate: 40.3%
  Top 10 most incomplete columns:
    Policyholders Equity                                    100.0%
    Income (Loss) from Affiliates, Net of Taxes              99.9%
    Change in Inventories                                    99.5%
    Change in Accounts Payable                               99.4%
    Change in Accounts Receivable                            99.3%
    Change in Other                                          99.1%
    Effect of Foreign Exchange Rates                         99.0%
    Preferred Equity                                         98.6%
    Total Claims & Losses                                    98.6%
    Insurance Reserves                                       98.5%


## 8. Save Raw Panel

In [18]:
df_fund.reset_index().to_csv('simfin_raw_panel.csv', index=False)
print(f"Saved simfin_raw_panel.csv — {df_fund.shape[0]:,} rows × {df_fund.shape[1]:,} columns")


Saved simfin_raw_panel.csv — 18,088 rows × 86 columns


In [19]:
df_fund

Fiscal Year       Revenue  Cost of Revenue  Gross Profit  \
Ticker Report Date                                                             
A      2020-10-31        2020.0  5.339000e+09    -2.502000e+09  2.837000e+09   
       2021-10-31        2021.0  6.319000e+09    -2.912000e+09  3.407000e+09   
       2022-10-31        2022.0  6.848000e+09    -3.126000e+09  3.722000e+09   
       2023-10-31        2023.0  6.833000e+09    -3.368000e+09  3.465000e+09   
       2024-10-31        2024.0  6.510000e+09    -2.975000e+09  3.535000e+09   
...                         ...           ...              ...           ...   
ZYXI   2022-12-31        2022.0  1.581670e+08    -3.200500e+07  1.261620e+08   
       2023-12-31        2023.0  1.843220e+08    -3.836600e+07  1.459560e+08   
       2024-12-31        2024.0  1.923540e+08    -3.942900e+07  1.529250e+08   
nan    2021-12-31        2021.0  6.112768e+06    -6.746630e+06 -6.338610e+05   
       2022-12-31        2022.0  1.461012e+08    -1.118981e+08  3.420300e+07   

                    Operating Expenses  Selling, General & Administrative  \
Ticker Report Date                                                          
A      2020-10-31        -1.991000e+09                      -1.496000e+09   
       2021-10-31        -2.060000e+09                      -1.619000e+09   
       2022-10-31        -2.104000e+09                      -1.637000e+09   
       2023-10-31        -2.115000e+09                      -1.634000e+09   
       2024-10-31        -2.047000e+09                      -1.568000e+09   
...                                ...                                ...   
ZYXI   2022-12-31        -1.032240e+08                      -1.032240e+08   
       2023-12-31        -1.351760e+08                      -1.351760e+08   
       2024-12-31        -1.469350e+08                      -1.469350e+08   
nan    2021-12-31        -2.707536e+07                      -4.531915e+06   
       2022-12-31        -3.112928e+07                      -1.153583e+07   

                    Research & Development  Depreciation & Amortization  \
Ticker Report Date                                                        
A      2020-10-31             -495000000.0                          NaN   
       2021-10-31             -441000000.0                          NaN   
       2022-10-31             -467000000.0                          NaN   
       2023-10-31             -481000000.0                          NaN   
       2024-10-31             -479000000.0                          NaN   
...                                    ...                          ...   
ZYXI   2022-12-31                      NaN                          NaN   
       2023-12-31                      NaN                          NaN   
       2024-12-31                      NaN                          NaN   
nan    2021-12-31              -22543449.0                          NaN   
       2022-12-31              -19593450.0                          NaN   

                    Operating Income (Loss)  Non-Operating Income (Loss)  ...  \
Ticker Report Date                                                        ...   
A      2020-10-31              8.460000e+08                   -4000000.0  ...   
       2021-10-31              1.347000e+09                   13000000.0  ...   
       2022-10-31              1.618000e+09                 -114000000.0  ...   
       2023-10-31              1.350000e+09                  -11000000.0  ...   
       2024-10-31              1.488000e+09                   33000000.0  ...   
...                                     ...                          ...  ...   
ZYXI   2022-12-31              2.293800e+07                    -440000.0  ...   
       2023-12-31              1.078000e+07                   -1094000.0  ...   
       2024-12-31              5.990000e+06                   -2382000.0  ...   
nan    2021-12-31             -2.770923e+07                     595361.0  ...   
       2022-12-31            

## 9. Compute 1-Year Forward Returns

In [20]:
price_col = 'Adj. Close' if 'Adj. Close' in df_prices.columns else 'Close'
prices_clean = df_prices[[price_col]].rename(columns={price_col: 'Price'})
prices_clean = prices_clean[prices_clean['Price'] > 0]

# Pre-build dict: ticker → Date-only index (Ticker level dropped)
prices_grouped = {
    ticker: grp.droplevel('Ticker').sort_index()
    for ticker, grp in prices_clean.groupby(level='Ticker')
}

def lookup_price(tp, target_date, max_lag=5):
    window = tp.loc[target_date : target_date + pd.Timedelta(days=max_lag)]
    return float(window['Price'].iloc[0]) if len(window) else np.nan

print("Computing 1-year forward returns...")
results = {}
for ticker, report_date in df_fund.index:
    tp = prices_grouped.get(str(ticker))
    if tp is None:
        results[(ticker, report_date)] = np.nan
        continue
    p0 = lookup_price(tp, report_date)
    p1 = lookup_price(tp, report_date + pd.Timedelta(days=252))
    results[(ticker, report_date)] = (p1 - p0) / p0 if (p0 and p1 and p0 != 0) else np.nan

df_fund['Perform'] = pd.Series(results)
print(f"Forward returns: {df_fund['Perform'].notna().sum():,} non-null / {len(df_fund):,} total")
print(df_fund['Perform'].describe().round(3))


Computing 1-year forward returns...
Forward returns: 13,601 non-null / 18,088 total
count    13601.000
mean         0.066
std          1.172
min         -1.000
25%         -0.247
50%         -0.007
75%          0.226
max         94.238
Name: Perform, dtype: float64


## 10. Construct Class Labels

In [21]:
df_fund['FiscalYear'] = df_fund.index.get_level_values('Report Date').year

def assign_class(df):
    def _tercile(group):
        r = group['Perform'].rank(method='first', na_option='keep', pct=True)
        return pd.cut(r, bins=[0, 1/3, 2/3, 1.0],
                      labels=[-1, 0, 1], include_lowest=True).astype(float)
    df = df.copy()
    df['Class'] = df.groupby('FiscalYear', group_keys=False).apply(_tercile)
    return df

df_fund = assign_class(df_fund)

print("Class distribution:")
print(df_fund['Class'].value_counts().sort_index())


Class distribution:
Class
-1.0    4531
 0.0    4534
 1.0    4536
Name: count, dtype: int64


/var/folders/40/7bxxvd3x4pj5vxgyywf02b6c0000gn/T/ipykernel_44835/1068342171.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df['Class'] = df.groupby('FiscalYear', group_keys=False).apply(_tercile)


## 11. Save Final Dataset

In [22]:
df_fund = df_fund.reset_index()
df_fund.to_csv('simfin_dataset.csv', index=False)
print(f"Saved simfin_dataset.csv")
print(f"Shape: {df_fund.shape}")
print(f"Unique companies: {df_fund['Ticker'].nunique():,}")


Saved simfin_dataset.csv
Shape: (18088, 91)
Unique companies: 4,787
